In [1]:
!pip install tdqm
!pip install evaluate
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for tdqm: filename=tdqm-0.0.1-py3-none-any.whl size=1384 sha256=a537f3512917258bea56e788c04740a3cfa7e9200951fa6bf199debac7e21cf8
  Stored in directory: /root/.cache/pip/wheels/9a/91/f3/44d4e5da089935024002c5351f851f6f96e3ffa76995154241
Successfully built tdqm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=ff30a70556cd3f675d2bc0635db717e60e2574d694ffff56f0edff5add0aa8b1
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [2]:
import gdown

url = "https://drive.google.com/uc?id=1CVMenH5xgDsiq9ZaGAFfkFLG5uWDZvdL"

output = "dataset.txt"

gdown.download(url, output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1CVMenH5xgDsiq9ZaGAFfkFLG5uWDZvdL
To: /content/dataset.txt
100%|██████████| 10.8M/10.8M [00:00<00:00, 48.3MB/s]


'dataset.txt'

In [3]:
import torch
from torch.utils.data import Dataset, random_split

class TranslationDataset(Dataset):
    def __init__(self, file_path, tokenizer_src, tokenizer_tgt, max_length=64):
        """
        file_path: đường dẫn đến file txt chứa 'src \t tgt'
        tokenizer_src, tokenizer_tgt: tokenizer pretrained
        """
        self.pairs = []
        with open(file_path, encoding="utf-8") as f:
            for line in f:
                if "\t" in line:
                    src, tgt = line.strip().split("\t")
                    self.pairs.append((src, tgt))

        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]

        src_enc = self.tokenizer_src(
            src_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        tgt_enc = self.tokenizer_tgt(
            tgt_text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": src_enc["input_ids"].squeeze(0),
            "attention_mask": src_enc["attention_mask"].squeeze(0),
            "labels": tgt_enc["input_ids"].squeeze(0),
        }


def load_translation_datasets(file_path, tokenizer_src, tokenizer_tgt, max_length=64):
    """
    Tạo dataset và chia train/val/test theo tỉ lệ 70/20/10.
    """
    full_dataset = TranslationDataset(file_path, tokenizer_src, tokenizer_tgt, max_length=max_length)

    n = len(full_dataset)
    n_train = int(0.7 * n)
    n_val = int(0.2 * n)
    n_test = n - n_train - n_val

    train_set, val_set, test_set = random_split(full_dataset, [n_train, n_val, n_test],
                                                generator=torch.Generator().manual_seed(42))
    return train_set, val_set, test_set

In [4]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

tokenizer_src = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
tokenizer_tgt = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

train_set, val_set, test_set = load_translation_datasets(
    "dataset.txt",
    tokenizer_src,
    tokenizer_tgt,
    max_length=64
)

train_loader = DataLoader(train_set, batch_size=4, shuffle=True)
val_loader = DataLoader(val_set, batch_size=4)
test_loader = DataLoader(test_set, batch_size=4)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")


config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Train: 106974 | Val: 30564 | Test: 15282


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer

class Seq2SeqTransformer(nn.Module):
    def __init__(self, num_encoder_layers, num_decoder_layers, emb_size,
                 nhead, src_vocab_size, tgt_vocab_size, dim_feedforward=512, dropout=0.1):
        super().__init__()

        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )

        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size)
        self.pos_encoder = PositionalEncoding(emb_size, dropout)
        self.pos_decoder = PositionalEncoding(emb_size, dropout)
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask,
                src_padding_mask, tgt_padding_mask, memory_key_padding_mask):
        src_emb = self.pos_encoder(self.src_tok_emb(src))
        tgt_emb = self.pos_decoder(self.tgt_tok_emb(tgt))
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)
        outs = self.transformer.decoder(
            tgt_emb, memory,
            tgt_mask=tgt_mask,
            memory_key_padding_mask=memory_key_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )
        return self.generator(outs)

    def encode(self, src, src_mask):
        return self.transformer.encoder(self.pos_encoder(self.src_tok_emb(src)), src_mask)

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(self.pos_decoder(self.tgt_tok_emb(tgt)), memory, tgt_mask)


class PositionalEncoding(nn.Module):
    def __init__(self, emb_size, dropout, maxlen=5000):
        super().__init__()
        den = torch.exp(- torch.arange(0, emb_size, 2) * (torch.log(torch.tensor(10000.0)) / emb_size))
        pos = torch.arange(0, maxlen).reshape(maxlen, 1)
        pos_embedding = torch.zeros((maxlen, emb_size))
        pos_embedding[:, 0::2] = torch.sin(pos * den)
        pos_embedding[:, 1::2] = torch.cos(pos * den)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('pos_embedding', pos_embedding)

    def forward(self, token_embedding):
        seq_len = token_embedding.size(0)
        pos_emb = self.pos_embedding[:seq_len, :].unsqueeze(1)  # [seq_len, 1, emb_dim]
        return self.dropout(token_embedding + pos_emb)


def generate_square_subsequent_mask(sz, device=None):
    mask = (torch.triu(torch.ones(sz, sz, device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask


def create_mask(src, tgt, pad_idx):
    src_seq_len = src.size(0)
    tgt_seq_len = tgt.size(0)
    device = src.device
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len).to(device)
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=device).type(torch.bool)
    src_padding_mask = (src == pad_idx).transpose(0, 1).to(device)
    tgt_padding_mask = (tgt == pad_idx).transpose(0, 1).to(device)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask



In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import evaluate


# ----------------- TRAINING -----------------
def train_epoch(model, dataloader, criterion, optimizer, pad_idx, device):
    model.train()
    total_loss = 0
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for batch in pbar:
        src = batch["input_ids"].to(device).transpose(0, 1)
        tgt = batch["labels"].to(device).transpose(0, 1)

        tgt_input = tgt[:-1, :]
        tgt_out = tgt[1:, :]

        src_mask, tgt_mask, src_pad_mask, tgt_pad_mask = create_mask(src, tgt_input, pad_idx)
        optimizer.zero_grad()

        logits = model(
            src, tgt_input, src_mask, tgt_mask,
            src_pad_mask, tgt_pad_mask, src_pad_mask
        )

        loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate_model(model, dataloader, criterion, pad_idx, tokenizer_tgt, device):
    model.eval()
    total_loss = 0.0
    preds, refs = [], []

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        # Lấy dữ liệu từ batch
        src = batch["input_ids"].to(device).transpose(0, 1)
        tgt = batch["labels"].to(device).transpose(0, 1)
        tgt_input = tgt[:-1, :]
        tgt_out = tgt[1:, :]

        # Tạo mask
        src_mask, tgt_mask, src_pad_mask, tgt_pad_mask = create_mask(src, tgt_input, pad_idx)

        # Forward qua model
        logits = model(
            src, tgt_input, src_mask, tgt_mask,
            src_pad_mask, tgt_pad_mask, src_pad_mask
        )

        # Tính loss
        loss = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        total_loss += loss.item()

        # Giải mã chuỗi dự đoán và nhãn thật
        predicted_ids = torch.argmax(logits, dim=-1).transpose(0, 1)
        for pred, ref in zip(predicted_ids, tgt_out.transpose(0, 1)):
            preds.append(tokenizer_tgt.decode(pred, skip_special_tokens=True))
            refs.append(tokenizer_tgt.decode(ref, skip_special_tokens=True))

    # Load metric
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")

    # Giữ nguyên cách xử lý chuỗi trong code cũ
    preds_str = [" ".join(p) if isinstance(p, list) else p for p in preds]
    refs_str = [" ".join(r) if isinstance(r, list) else r for r in refs]

    bleu_result = bleu.compute(predictions=preds_str, references=refs_str)
    rouge_result = rouge.compute(predictions=preds_str, references=refs_str)

    if isinstance(rouge_result["rougeL"], dict) and "mid" in rouge_result["rougeL"]:
        rougeL_score = rouge_result["rougeL"]["mid"]["fmeasure"]
    elif hasattr(rouge_result["rougeL"], "mid"):
        rougeL_score = rouge_result["rougeL"].mid.fmeasure
    else:
        rougeL_score = rouge_result["rougeL"]

    avg_loss = total_loss / len(dataloader)
    return avg_loss, bleu_result["bleu"], rougeL_score


def main_train(train_loader, val_loader, test_loader, tokenizer_src, tokenizer_tgt, pad_idx):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = Seq2SeqTransformer(
        num_encoder_layers=3,
        num_decoder_layers=3,
        emb_size=256,
        nhead=8,
        src_vocab_size=tokenizer_src.vocab_size,
        tgt_vocab_size=tokenizer_tgt.vocab_size,
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

    train_losses, val_losses = [], []

    for epoch in range(1, 11):
        print(f"\nEpoch {epoch}/10")
        train_loss = train_epoch(model, train_loader, criterion, optimizer, pad_idx, device)
        val_loss, bleu, rouge = evaluate_model(model, val_loader, criterion, pad_idx, tokenizer_tgt, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | BLEU: {bleu:.4f} | ROUGE-L: {rouge:.4f}")

    # Evaluate trên test set
    test_loss, bleu, rouge = evaluate_model(model, test_loader, criterion, pad_idx, tokenizer_tgt, device)
    print(f"\nTest loss: {test_loss:.4f} | BLEU: {bleu:.4f} | ROUGE-L: {rouge:.4f}")

    # Plot loss
    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Training & Validation Loss")
    plt.show()

In [7]:
if __name__ == "__main__":

    tokenizer_src = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")
    tokenizer_tgt = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

    train_set, val_set, test_set = load_translation_datasets("dataset.txt", tokenizer_src, tokenizer_tgt)
    pad_idx = tokenizer_src.pad_token_id

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=128)
    test_loader = DataLoader(test_set, batch_size=128)

    main_train(train_loader, val_loader, test_loader, tokenizer_src, tokenizer_tgt, pad_idx)

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(



Epoch 1/10


Training:   0%|          | 0/836 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


KeyboardInterrupt: 

In [ ]:
# Do resource có hạn nên chạy các iteration đầu ạ